# Dental expert-model pipeline — Kaggle deployment and experiment runner

This notebook is the **single runner/orchestrator notebook** for the project.

Runtime architecture:

```text
Panoramic X-ray
      ↓
llama.cpp multimodal server
      ├── DentalGPT-7B-1026 GGUF
      └── DentalGPT mmproj GGUF
      ↓
DentalExpertModelRunner.ask(image, question)
      ↓
BASIC / DISEASE_HIERARCHY / DISEASE_AND_LOCATION
      ↓
raw observations + deterministic atomic statuses
      ↓
optional text-only orchestrator: dentist report, then evaluation adaptation
      ↓
saved JSON
```

Important design choices:

- Do **not** separately load `Qwen/Qwen2.5-VL-7B-Instruct`.
- Do **not** use Transformers or BitsAndBytes for the DentalGPT GGUF.
- The current dental expert-model deployment is DentalGPT; the pipeline role remains replaceable.
- Keep dentist-report synthesis and evaluation adaptation outside the expert model.
- Start with `BASIC`, verify one real run, then move to deeper modes.


In [ ]:
# ============================================================
# CELL 1 — Python dependencies
# ============================================================
# llama.cpp itself is compiled later with CUDA.
# We intentionally do not install transformers / bitsandbytes / qwen-vl-utils.

%pip install -q \
    "huggingface_hub>=0.26" \
    "openai>=1.55" \
    "pydantic>=2.7" \
    "PyYAML>=6.0" \
    "requests>=2.31" \
    "pillow>=10.0"

print("Python dependencies installed.")


In [ ]:
# ============================================================
# CELL 2 — Locate/import the project files
# ============================================================
# Supported Kaggle layouts:
# A) /kaggle/working/dentalgpt_project_rewrite
# B) dentalgpt_project_rewrite.zip added as a Kaggle dataset
# C) current directory contains the required Python files

import os
import sys
import json
import time
import shutil
import zipfile
import subprocess
from pathlib import Path

REQUIRED_PROJECT_FILES = {
    "dentalgpt.py",
    "benchmark.py",
    "evaluation.py",
    "llama_runtime.py",
    "pipeline.py",
    "prompts.py",
}

WORK_PROJECT_DIR = Path("/kaggle/working/dentalgpt_project_rewrite")


def has_project_files(path: Path) -> bool:
    return path.is_dir() and REQUIRED_PROJECT_FILES.issubset(
        {p.name for p in path.iterdir() if p.is_file()}
    )


project_candidates = [WORK_PROJECT_DIR, Path.cwd()]
PROJECT_DIR = next((p for p in project_candidates if has_project_files(p)), None)

# If modules are not directly available, try the project ZIP from /kaggle/input.
if PROJECT_DIR is None and Path("/kaggle/input").exists():
    zip_hits = list(Path("/kaggle/input").rglob("dentalgpt_project_rewrite.zip"))
    if zip_hits:
        project_zip = zip_hits[0]
        print("Found project ZIP:", project_zip)
        with zipfile.ZipFile(project_zip, "r") as zf:
            zf.extractall("/kaggle/working")
        if has_project_files(WORK_PROJECT_DIR):
            PROJECT_DIR = WORK_PROJECT_DIR

# Last fallback: search /kaggle/input for the modules themselves.
if PROJECT_DIR is None and Path("/kaggle/input").exists():
    for dentalgpt_file in Path("/kaggle/input").rglob("dentalgpt.py"):
        candidate = dentalgpt_file.parent
        if has_project_files(candidate):
            PROJECT_DIR = candidate
            break

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not locate the project modules. Add dentalgpt_project_rewrite.zip "
        "to the Kaggle notebook as a dataset, or place all required Python modules under "
        "/kaggle/working/dentalgpt_project_rewrite."
    )

PROJECT_DIR = PROJECT_DIR.resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("PROJECT_DIR =", PROJECT_DIR)

from dentalgpt import DentalExpertModelRunner
from benchmark import VisionLocationResolver, load_yolo_benchmark, prepare_vision_location_cache
from evaluation import EvaluationConfig, compare_experiments, evaluate_experiment
from llama_runtime import LlamaCppServer, build_llama_cpp, download_dentalgpt, find_llama_server
from pipeline import DentalAnalysisPipeline, LLMOrchestrator
from prompts import broad_records

print("Project imports succeeded.")


In [ ]:
# ============================================================
# CELL 3 — MAIN EXPERIMENT CONFIGURATION
# ============================================================
# This is the main cell to edit between experiments.

# Output
OUTPUT_DIR = "/kaggle/working/dental_outputs"

# BASIC: 4 broad screening calls.
# DISEASE_HIERARCHY: broad + family + 14 atomic calls.
# DISEASE_AND_LOCATION: hierarchy + location follow-ups.
ANALYSIS_MODE = "BASIC"

RUN_SMOKE_TEST = True
RUN_PIPELINE = True
SHOW_IMAGE = True

# DentalGPT repository
HF_REPO_ID = "mradermacher/DentalGPT-7B-1026-GGUF"
MODEL_DIR = "/kaggle/working/models/dentalgpt"

# QUALITY = Q6_K + F16 mmproj, preferred baseline on a 16 GB P100.
# FAST    = Q4_K_M + Q8 mmproj, faster/lower-memory development preset.
MODEL_PRESET = "QUALITY"  # "QUALITY" | "FAST"

if MODEL_PRESET.upper() == "QUALITY":
    MODEL_FILENAME = "DentalGPT-7B-1026.Q6_K.gguf"
    MMPROJ_FILENAME = "DentalGPT-7B-1026.mmproj-f16.gguf"
elif MODEL_PRESET.upper() == "FAST":
    MODEL_FILENAME = "DentalGPT-7B-1026.Q4_K_M.gguf"
    MMPROJ_FILENAME = "DentalGPT-7B-1026.mmproj-Q8_0.gguf"
else:
    raise ValueError("MODEL_PRESET must be QUALITY or FAST")

# llama.cpp runtime
LLAMA_CPP_DIR = "/kaggle/working/llama.cpp"
LLAMA_CPP_REF = "b10516"
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8080
SERVER_ALIAS = "dentalgpt"
SERVER_LOG_PATH = "/kaggle/working/llama_dentalgpt_server.log"
N_GPU_LAYERS = 999
CTX_SIZE = 8192
PARALLEL = 1
CUDA_ARCH = None  # None = auto-detect; P100 fallback is 60
BUILD_JOBS = 4
SERVER_STARTUP_TIMEOUT = 300.0

# DentalGPT generation defaults
DEFAULT_MAX_TOKENS = 768
TEMPERATURE = 0.0
TOP_P = 1.0
SEED = 0
REQUEST_TIMEOUT_SECONDS = 600.0
CACHE_PROMPT = False
LOCATE_UNCERTAIN = True

# Optional external text-only orchestrator
USE_OPENAI_ORCHESTRATOR = False
ORCHESTRATOR_MODEL = None
ORCHESTRATOR_BASE_URL = None
ORCHESTRATOR_TIMEOUT_SECONDS = 600.0
ORCHESTRATOR_MAX_RETRIES = 2

# Optional offline benchmark evaluation. Keep disabled for ordinary single-image runs.
RUN_EVALUATION = False
BENCHMARK_IMAGES_DIR = None
BENCHMARK_LABELS_DIR = None
BENCHMARK_DATA_YAML = None
# Evaluation subset control (zero-based positions in sorted benchmark image IDs).
# Leave both as None for the full test set. Use only one selector at a time.
EVALUATION_SAMPLE_SIZE = None  # e.g. 5 for a reproducible random sample
EVALUATION_IMAGE_INDICES = None  # e.g. [0, 7, 12] for exact cases
EVALUATION_RANDOM_SEED = 0
EVALUATION_PREDICTIONS_DIR = OUTPUT_DIR
EVALUATION_RESULTS_PATH = f"{OUTPUT_DIR}/evaluation_results.json"
EXPERIMENT_NAME = "broad_v1"
EXPERIMENT_METADATA = {}  # e.g. prompt version/hash, agent structure, seed
EVALUATE_LOCATION = False
LOCATION_LEVEL = 0  # 0=off, 1=arch+side, 2=arch+side+anterior/posterior
LOCATION_ADAPTERS = ("vision",)  # ("geometry",), ("vision",), or both
LOCATION_VISION_BACKEND = "llm"  # "llm" or "expert_model"
LOCATION_VISION_MODEL = ORCHESTRATOR_MODEL
VISION_LOCATION_CACHE_PATH = f"{OUTPUT_DIR}/vision_location_cache.json"
ANNOTATED_BOXES_DIR = f"{OUTPUT_DIR}/annotated_boxes"
COMPARISON_RESULT_PATHS = []


print("Configuration:")
print("  mode         =", ANALYSIS_MODE)
print("  preset       =", MODEL_PRESET)
print("  model        =", MODEL_FILENAME)
print("  mmproj       =", MMPROJ_FILENAME)
print("  context      =", CTX_SIZE)
print("  orchestrator =", USE_OPENAI_ORCHESTRATOR)


In [ ]:
# ============================================================
# CELL 4 — Environment diagnostics
# ============================================================

import platform

print("Python:", platform.python_version())
print("Platform:", platform.platform())

for executable in ["git", "cmake", "nvcc", "nvidia-smi"]:
    print(f"{executable:12s}:", shutil.which(executable))

print("\nGPU:")
subprocess.run(["nvidia-smi"], check=False)


def detect_cuda_arch(fallback: str = "60") -> str:
    try:
        output = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            text=True,
            stderr=subprocess.STDOUT,
        )
        first = output.strip().splitlines()[0].strip()
        arch = first.replace(".", "")
        if arch.isdigit():
            return arch
    except Exception as exc:
        print("CUDA architecture auto-detection failed:", exc)

    print(f"Falling back to CUDA architecture {fallback}.")
    return fallback


CUDA_ARCH_RESOLVED = str(CUDA_ARCH) if CUDA_ARCH else detect_cuda_arch("60")
print("\nCUDA_ARCH_RESOLVED =", CUDA_ARCH_RESOLVED)

if not shutil.which("cmake"):
    raise RuntimeError("cmake is required to build llama.cpp.")
if not shutil.which("git"):
    raise RuntimeError("git is required to obtain llama.cpp.")
if not shutil.which("nvcc"):
    raise RuntimeError("nvcc was not found. Enable a GPU accelerator in Kaggle.")


In [ ]:
# ============================================================
# CELL 5 — Kaggle secrets
# ============================================================
# HF_TOKEN is normally optional because the GGUF repo is public.
# OPENAI_API_KEY is needed only for the optional orchestrator.

HF_TOKEN = HF_TOKEN
ORCHESTRATOR_API_KEY = ORCHESTRATOR_API_KEY

if USE_OPENAI_ORCHESTRATOR:
    if not ORCHESTRATOR_MODEL:
        raise ValueError("Set ORCHESTRATOR_MODEL when USE_OPENAI_ORCHESTRATOR=True.")
    if not ORCHESTRATOR_API_KEY:
        raise ValueError("OPENAI_API_KEY is required when USE_OPENAI_ORCHESTRATOR=True.")

print("HF token configured:", bool(HF_TOKEN))
print("External orchestrator enabled:", USE_OPENAI_ORCHESTRATOR)


In [ ]:
# ============================================================
# CELL 6 — Build/find pinned llama.cpp with CUDA
# ============================================================

LLAMA_SERVER = find_llama_server()

if LLAMA_SERVER is None:
    print("llama-server not found; building pinned llama.cpp...")
    LLAMA_SERVER = build_llama_cpp(
        source_dir=LLAMA_CPP_DIR,
        cuda_arch=CUDA_ARCH_RESOLVED,
        jobs=BUILD_JOBS,
        ref=LLAMA_CPP_REF,
    )
else:
    print("Found existing llama-server:", LLAMA_SERVER)

LLAMA_SERVER = Path(LLAMA_SERVER).resolve()
if not LLAMA_SERVER.is_file():
    raise FileNotFoundError(LLAMA_SERVER)

print("llama-server =", LLAMA_SERVER)


In [ ]:
# ============================================================
# CELL 7 — Download the exact DentalGPT GGUF + mmproj
# ============================================================

model_files = download_dentalgpt(
    model_dir=MODEL_DIR,
    repo_id=HF_REPO_ID,
    model_filename=MODEL_FILENAME,
    mmproj_filename=MMPROJ_FILENAME,
    hf_token=HF_TOKEN,
)

MODEL_PATH = Path(model_files.model_path).resolve()
MMPROJ_PATH = Path(model_files.mmproj_path).resolve()

print("Language model:")
print(" ", MODEL_PATH)
print(f"  size = {MODEL_PATH.stat().st_size / (1024**3):.2f} GiB")

print("\nVision projector:")
print(" ", MMPROJ_PATH)
print(f"  size = {MMPROJ_PATH.stat().st_size / (1024**3):.2f} GiB")


In [ ]:
# ============================================================
# CELL 8 — Start a clean DentalGPT llama.cpp server
# ============================================================
# Rerunning this cell first stops the server object owned by this notebook.
# Then /v1/models is checked so we do not accidentally use another model.

import requests

if "server" in globals():
    try:
        server.stop()
    except Exception as exc:
        print("Previous server cleanup:", exc)

server = LlamaCppServer(
    binary=LLAMA_SERVER,
    model_path=MODEL_PATH,
    mmproj_path=MMPROJ_PATH,
    host=SERVER_HOST,
    port=SERVER_PORT,
    alias=SERVER_ALIAS,
    n_gpu_layers=N_GPU_LAYERS,
    ctx_size=CTX_SIZE,
    parallel=PARALLEL,
    startup_timeout=SERVER_STARTUP_TIMEOUT,
    log_path=SERVER_LOG_PATH,
)

server.start(reuse_existing=False)

models_response = requests.get(f"{server.base_url}/v1/models", timeout=10)
models_response.raise_for_status()
models_payload = models_response.json()
model_ids = [
    item.get("id")
    for item in models_payload.get("data", [])
    if isinstance(item, dict)
]

print("Server URL:", server.base_url)
print("Server model IDs:", model_ids)

if SERVER_ALIAS not in model_ids:
    raise RuntimeError(
        f"Expected alias {SERVER_ALIAS!r}, but /v1/models returned {model_ids}. "
        f"Inspect {SERVER_LOG_PATH}."
    )

print("DentalGPT llama.cpp server verified.")


In [ ]:
# ============================================================
# CELL 9 — Construct the dental expert-model runner
# ============================================================

expert_model_runner = DentalExpertModelRunner(
    base_url=server.base_url,
    api_model=SERVER_ALIAS,
    model_id=f"{HF_REPO_ID}:{MODEL_FILENAME}",
    max_tokens=DEFAULT_MAX_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    seed=SEED,
    timeout=REQUEST_TIMEOUT_SECONDS,
    cache_prompt=CACHE_PROMPT,
)

print("Dental expert-model runner ready.")
print("model_id =", expert_model_runner.model_id)


In [ ]:
# ============================================================
# CELL 9.5 - Choose the image and its matching YOLO label
# ============================================================
# Change these two paths for each single-image experiment.

IMAGE_PATH = "/kaggle/input/YOUR_DATASET/images/YOUR_IMAGE.jpg"
LABEL_FILE_PATH = "/kaggle/input/YOUR_DATASET/labels/YOUR_IMAGE.txt"

print("IMAGE_PATH      =", IMAGE_PATH)
print("LABEL_FILE_PATH =", LABEL_FILE_PATH)


In [ ]:
# ============================================================
# CELL 10 — Validate and preview the input radiograph
# ============================================================

from PIL import Image
from IPython.display import display

image_path = Path(IMAGE_PATH)
if not image_path.is_file():
    raise FileNotFoundError(
        f"IMAGE_PATH does not exist:\n{image_path}\n\n"
        "Edit IMAGE_PATH in CELL 9.5 before continuing."
    )

image = Image.open(image_path)
print("Image:", image_path)
print("Format:", image.format)
print("Mode:", image.mode)
print("Size:", image.size)

if SHOW_IMAGE:
    display(image)


In [ ]:
# ============================================================
# CELL 11 — One real production-prompt smoke test
# ============================================================
# Verifies image encoding, mmproj, multimodal formatting and generation.
# Uses the first real BASIC prompt rather than a separate demo prompt.

smoke = None

if RUN_SMOKE_TEST:
    smoke_record = broad_records()[0]

    print("question_id:", smoke_record["question_id"])
    print("layer:", smoke_record["layer"])
    print("\nQUESTION\n--------")
    print(smoke_record["question"])

    smoke = expert_model_runner.ask(
        IMAGE_PATH,
        smoke_record["question"],
        max_tokens=smoke_record.get("max_tokens"),
    )

    print("\nRAW EXPERT-MODEL RESPONSE\n-------------------------")
    print(smoke["raw_answer"])

    print("\nMETADATA")
    print("  latency_seconds    =", smoke.get("latency_seconds"))
    print("  finish_reason      =", smoke.get("finish_reason"))
    print("  truncated          =", smoke.get("truncated"))
    print("  prompt_tokens      =", smoke.get("prompt_tokens"))
    print("  completion_tokens  =", smoke.get("completion_tokens"))

    if smoke.get("truncated"):
        print(
            "\nWARNING: Smoke test hit the token limit. "
            "Increase that prompt's max_tokens before interpreting its answer."
        )
else:
    print("RUN_SMOKE_TEST=False; skipped.")


In [ ]:
# ============================================================
# CELL 12 — Build the analysis pipeline
# ============================================================

orchestrator = None
if USE_OPENAI_ORCHESTRATOR:
    orchestrator = LLMOrchestrator(
        model=ORCHESTRATOR_MODEL,
        base_url=ORCHESTRATOR_BASE_URL,
        api_key=ORCHESTRATOR_API_KEY,
        timeout=ORCHESTRATOR_TIMEOUT_SECONDS,
        max_retries=ORCHESTRATOR_MAX_RETRIES,
    )

pipeline = DentalAnalysisPipeline(
    expert_model_runner=expert_model_runner,
    orchestrator=orchestrator,
    locate_uncertain=LOCATE_UNCERTAIN,
)

print("Pipeline ready.")
print("Analysis mode:", ANALYSIS_MODE)
print("Locate UNCERTAIN findings:", LOCATE_UNCERTAIN)
print("External orchestrator:", bool(orchestrator))


In [ ]:
# ============================================================
# CELL 13 — Orchestrator connectivity smoke test
# ============================================================
# This sends text only and does not run the dental expert model or pipeline.

if orchestrator is None:
    print("External orchestrator is disabled; skipped.")
else:
    smoke_completion = orchestrator.client.chat.completions.create(
        model=orchestrator.model,
        messages=[{"role": "user", "content": "Reply with exactly: ORCHESTRATION_OK"}],
    )
    print("Orchestrator smoke response:", smoke_completion.choices[0].message.content)


In [ ]:
# ============================================================
# CELL 14 — Run the selected analysis mode
# ============================================================

result = None

if RUN_PIPELINE:
    result = pipeline.run(
        image_path=IMAGE_PATH,
        mode=ANALYSIS_MODE,
        output_dir=OUTPUT_DIR,
    )

    print("\nRUN COMPLETE")
    print("------------")
    print("Saved to:", result["saved_to"])
    print("Dental expert model:", result["expert_model"])
    print("Expert-model calls:", result["expert_model_call_count"])
    print("Total latency:", result["total_latency_seconds"], "seconds")
else:
    print("RUN_PIPELINE=False; skipped.")


In [ ]:
# ============================================================
# CELL 15 — Compact result summary
# ============================================================

import pandas as pd
from IPython.display import display

if result is None:
    print("No pipeline result. Run CELL 13 first.")
else:
    atomic_rows = []
    for item in result["observations"]:
        if item.get("layer") == "ATOMIC_FINDING":
            atomic_rows.append(
                {
                    "condition": item.get("target"),
                    "status": item.get("parsed_status"),
                    "latency_s": item.get("latency_seconds"),
                    "finish_reason": item.get("finish_reason"),
                    "truncated": item.get("truncated"),
                }
            )

    if atomic_rows:
        print("Atomic findings:")
        display(pd.DataFrame(atomic_rows))
    else:
        print("No atomic findings in this run. That is expected in BASIC mode.")

    location_rows = []
    for item in result["observations"]:
        if item.get("layer") == "LOCATION":
            location_rows.append(
                {
                    "condition": item.get("target"),
                    "question_id": item.get("question_id"),
                    "answer": item.get("parsed_answer"),
                    "latency_s": item.get("latency_seconds"),
                    "truncated": item.get("truncated"),
                }
            )

    if location_rows:
        print("\nLocation follow-ups:")
        display(pd.DataFrame(location_rows))

    if result.get("dentist_report"):
        print("\nDentist report:")
        print(result["dentist_report"]["report"])
        print("\nEvaluation adaptation report:")
        print(json.dumps(result["evaluation_adaptation_report"], indent=2))


In [ ]:
# ============================================================
# CELL 15.1 - Direct orchestrator inference
# ============================================================
# This is the text-only equivalent of expert_model_runner.ask(...).
# Change the input and prompt directly for each research experiment.

RUN_ORCHESTRATOR_INFERENCE = False
ORCHESTRATOR_INFERENCE_MAX_TOKENS = 1200
ORCHESTRATOR_INFERENCE_TEMPERATURE = 0.0

question_orchestrator = """
You are a dental report checker. Preserve supported findings, point out conflicts,
and return a clear corrected report. Do not claim to see the radiograph.
""".strip()

orchestrator_input = (
    smoke["raw_answer"]
    if smoke is not None
    else "Paste an FDM response, observation list, or report here."
)

orchestrator_inference = None

if RUN_ORCHESTRATOR_INFERENCE:
    if orchestrator is None:
        raise RuntimeError(
            "Enable and build the orchestrator in CELLS 3, 5, and 12 first."
        )

    orchestrator_inference = orchestrator.ask(
        orchestrator_input,
        question_orchestrator,
        max_tokens=ORCHESTRATOR_INFERENCE_MAX_TOKENS,
        temperature=ORCHESTRATOR_INFERENCE_TEMPERATURE,
    )

    print("\nRAW ORCHESTRATOR RESPONSE\n-------------------------")
    print(orchestrator_inference["raw_answer"])
else:
    print("RUN_ORCHESTRATOR_INFERENCE=False; skipped.")


In [ ]:
# ============================================================
# CELL 15.2 - Read one YOLO label as finding names + locations
# ============================================================
# Each location tuple is: (x_center, y_center, width, height), normalized 0..1.
# If one finding appears several times, all of its boxes stay in the list.

from prompts import CONDITIONS

LABEL_CLASS_NAMES = {class_id: name for class_id, name in enumerate(CONDITIONS)}


def read_single_yolo_label(label_file_path, class_names=LABEL_CLASS_NAMES):
    label_path = Path(label_file_path)
    if not label_path.is_file():
        raise FileNotFoundError(label_path)

    decoded = {}
    for line_number, raw_line in enumerate(
        label_path.read_text(encoding="utf-8").splitlines(), start=1
    ):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split()
        if len(parts) != 5:
            raise ValueError(
                f"{label_path}:{line_number} must contain: "
                "class_id x_center y_center width height"
            )

        try:
            class_id = int(parts[0])
            location = tuple(float(value) for value in parts[1:])
        except ValueError as exc:
            raise ValueError(f"Invalid number at {label_path}:{line_number}") from exc

        if class_id not in class_names:
            raise ValueError(
                f"Unknown class ID {class_id} at {label_path}:{line_number}. "
                f"Expected one of {sorted(class_names)}."
            )
        if any(value < 0.0 or value > 1.0 for value in location):
            raise ValueError(
                f"Coordinates must be normalized to 0..1 at {label_path}:{line_number}."
            )

        finding_name = class_names[class_id]
        decoded.setdefault(finding_name, []).append(location)

    return decoded


ground_truth_label = None
if Path(LABEL_FILE_PATH).is_file():
    ground_truth_label = read_single_yolo_label(LABEL_FILE_PATH)
    print("Tuple format: (x_center, y_center, width, height)")
    print(json.dumps(ground_truth_label, ensure_ascii=False, indent=2))
else:
    print("Set LABEL_FILE_PATH before running the evaluation phase.")


In [ ]:
# ============================================================
# CELL 15.3 - Direct evaluation-phase inference
# ============================================================
# Change evaluation_input and question_evaluation directly. For example, evaluate:
#   orchestrator_inference["raw_answer"]
#   smoke["raw_answer"]
#   result["dentist_report"]
#   result

RUN_EVALUATION_INFERENCE = False
EVALUATION_INFERENCE_MAX_TOKENS = 1800
EVALUATION_INFERENCE_TEMPERATURE = 0.0

GROUND_TRUTH_IS_EXHAUSTIVE = True  # False for partially annotated labels.

question_evaluation = """
You evaluate a dental-model or dental-orchestrator answer against YOLO ground truth.
The ground truth lists positive findings as normalized (x_center, y_center, width, height)
boxes. Use ground_truth_is_exhaustive to decide whether an unlisted ontology class is absent
or simply unannotated. Judge finding detection separately from location. Do not punish a
correct finding only because its location is wrong. If textual and box locations cannot be
reliably aligned, say location is not assessable instead of guessing.
Be strict about unsupported claims and uncertainty, and explain ambiguous cases.

Return only valid JSON with these keys:
overall_assessment, correctly_found, missed_findings, extra_or_unsupported_findings,
location_assessment, uncertainty_notes, strengths, weaknesses, recommended_prompt_changes.
Use lists where several items are possible.
""".strip()

evaluation_input = {
        "closed_ontology": list(CONDITIONS),
        "ground_truth": ground_truth_label,
        "ground_truth_is_exhaustive": GROUND_TRUTH_IS_EXHAUSTIVE,
        "inference_output": (
            orchestrator_inference["raw_answer"]
            if orchestrator_inference is not None
            else smoke["raw_answer"] if smoke is not None else "Paste an output here."
        ),
}

evaluation_inference = None

if RUN_EVALUATION_INFERENCE:
    if orchestrator is None:
        raise RuntimeError(
            "Enable and build the orchestrator in CELLS 3, 5, and 12 first."
        )
    if ground_truth_label is None:
        ground_truth_label = read_single_yolo_label(LABEL_FILE_PATH)
        evaluation_input["ground_truth"] = ground_truth_label

    evaluation_inference = orchestrator.ask(
        evaluation_input,
        question_evaluation,
        max_tokens=EVALUATION_INFERENCE_MAX_TOKENS,
        temperature=EVALUATION_INFERENCE_TEMPERATURE,
    )

    print("\nRAW EVALUATION RESPONSE\n-----------------------")
    print(evaluation_inference["raw_answer"])
else:
    print("RUN_EVALUATION_INFERENCE=False; skipped.")


In [ ]:
# ============================================================
# CELL 16 — Optional offline benchmark evaluation and comparison
# ============================================================
# Evaluate a directory containing one saved pipeline JSON per benchmark image.
# Use a separate EVALUATION_PREDICTIONS_DIR for each prompt/system experiment.

import hashlib
import random

import pandas as pd
from IPython.display import display


def select_evaluation_benchmark(full_benchmark, sample_size=None, image_indices=None, seed=0):
    if sample_size is not None and image_indices is not None:
        raise ValueError("Set only one of EVALUATION_SAMPLE_SIZE or EVALUATION_IMAGE_INDICES.")

    all_image_ids = list(full_benchmark.image_ids)
    if image_indices is not None:
        indices = list(image_indices)
        if not indices:
            raise ValueError("EVALUATION_IMAGE_INDICES cannot be empty. Use None for all images.")
        if any(isinstance(index, bool) or not isinstance(index, int) for index in indices):
            raise TypeError("EVALUATION_IMAGE_INDICES must contain only integer positions.")
        if len(indices) != len(set(indices)):
            raise ValueError("EVALUATION_IMAGE_INDICES contains duplicate positions.")
        invalid = [index for index in indices if index < 0 or index >= len(all_image_ids)]
        if invalid:
            raise IndexError(
                f"Evaluation indices out of range: {invalid}; valid range is 0..{len(all_image_ids) - 1}."
            )
        selected_ids = [all_image_ids[index] for index in indices]
    elif sample_size is not None:
        if isinstance(sample_size, bool) or not isinstance(sample_size, int):
            raise TypeError("EVALUATION_SAMPLE_SIZE must be an integer or None.")
        if sample_size < 1 or sample_size > len(all_image_ids):
            raise ValueError(
                f"EVALUATION_SAMPLE_SIZE must be between 1 and {len(all_image_ids)}."
            )
        selected_ids = random.Random(seed).sample(all_image_ids, sample_size)
    else:
        return full_benchmark

    selected_images = {image_id: full_benchmark.images[image_id] for image_id in selected_ids}
    dataset_hash = hashlib.sha256()
    for image_id in sorted(selected_images):
        dataset_hash.update(image_id.encode("utf-8"))
        dataset_hash.update(selected_images[image_id].source_fingerprint.encode("ascii"))
    return full_benchmark.__class__(
        images=selected_images,
        fingerprint=dataset_hash.hexdigest(),
        images_dir=full_benchmark.images_dir,
        labels_dir=full_benchmark.labels_dir,
    )


def load_selected_predictions(prediction_dir, selected_image_ids):
    selected = set(selected_image_ids)
    predictions = {}
    prediction_keys = {
        "evaluation_adaptation_report", "findings",
        "deterministic_atomic_statuses", "statuses",
    }
    for path in sorted(Path(prediction_dir).rglob("*.json")):
        payload = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(payload, dict) or not prediction_keys.intersection(payload):
            continue
        raw_image_id = payload.get("image_id")
        if isinstance(raw_image_id, str) and raw_image_id:
            image_id = raw_image_id
        elif isinstance(payload.get("image_path"), str):
            image_id = Path(payload["image_path"]).stem
        else:
            raise ValueError(f"Prediction file {path} has no image_id or image_path.")
        if image_id not in selected:
            continue
        if image_id in predictions:
            raise ValueError(f"Duplicate prediction for selected image ID {image_id!r}.")
        predictions[image_id] = payload

    missing = selected - set(predictions)
    if missing:
        raise ValueError(f"Missing predictions for selected image IDs: {sorted(missing)}.")
    return predictions


evaluation_result = None

if RUN_EVALUATION:
    if not BENCHMARK_IMAGES_DIR or not BENCHMARK_LABELS_DIR:
        raise ValueError("Set BENCHMARK_IMAGES_DIR and BENCHMARK_LABELS_DIR.")

    full_benchmark = load_yolo_benchmark(
        BENCHMARK_IMAGES_DIR,
        BENCHMARK_LABELS_DIR,
        data_yaml=BENCHMARK_DATA_YAML,
    )
    benchmark = select_evaluation_benchmark(
        full_benchmark,
        sample_size=EVALUATION_SAMPLE_SIZE,
        image_indices=EVALUATION_IMAGE_INDICES,
        seed=EVALUATION_RANDOM_SEED,
    )
    selected_positions = [
        list(full_benchmark.image_ids).index(image_id) for image_id in benchmark.image_ids
    ]
    print(f"Evaluating {len(benchmark.image_ids)} of {len(full_benchmark.image_ids)} images.")
    print("Selected zero-based indices:", selected_positions)
    print("Selected image IDs:", list(benchmark.image_ids))

    evaluation_predictions = EVALUATION_PREDICTIONS_DIR
    if len(benchmark.image_ids) != len(full_benchmark.image_ids):
        evaluation_predictions = load_selected_predictions(
            EVALUATION_PREDICTIONS_DIR, benchmark.image_ids
        )
    evaluation_config = EvaluationConfig(
        evaluate_findings=True,
        evaluate_location=EVALUATE_LOCATION,
        location_level=LOCATION_LEVEL,
        location_adapters=tuple(LOCATION_ADAPTERS),
        vision_backend=LOCATION_VISION_BACKEND,
    )

    vision_cache = None
    if EVALUATE_LOCATION and "vision" in LOCATION_ADAPTERS:
        if LOCATION_VISION_BACKEND == "llm":
            if not LOCATION_VISION_MODEL:
                raise ValueError("Set LOCATION_VISION_MODEL for the llm vision backend.")
            location_resolver = VisionLocationResolver.from_openai_compatible(
                model=LOCATION_VISION_MODEL,
                base_url=ORCHESTRATOR_BASE_URL,
                api_key=ORCHESTRATOR_API_KEY,
                timeout=ORCHESTRATOR_TIMEOUT_SECONDS,
                max_retries=ORCHESTRATOR_MAX_RETRIES,
            )
        elif LOCATION_VISION_BACKEND == "expert_model":
            location_resolver = VisionLocationResolver.from_expert_model(expert_model_runner)
        else:
            raise ValueError("LOCATION_VISION_BACKEND must be llm or expert_model.")
        vision_cache = prepare_vision_location_cache(
            benchmark,
            location_resolver,
            VISION_LOCATION_CACHE_PATH,
            ANNOTATED_BOXES_DIR,
        )

    evaluation_result = evaluate_experiment(
        benchmark,
        evaluation_predictions,
        config=evaluation_config,
        experiment_metadata={
            "name": EXPERIMENT_NAME,
            "analysis_mode": ANALYSIS_MODE,
            "expert_model": MODEL_FILENAME,
            **EXPERIMENT_METADATA,
        },
        vision_cache=vision_cache,
        output_path=EVALUATION_RESULTS_PATH,
    )
    print("Evaluation saved to:", EVALUATION_RESULTS_PATH)
    display(pd.DataFrame([evaluation_result["overall_metrics"]]))
    display(
        pd.DataFrame.from_dict(evaluation_result["per_class_metrics"], orient="index")
        .rename_axis("condition")
        .reset_index()
    )
else:
    print("RUN_EVALUATION=False; skipped.")

if COMPARISON_RESULT_PATHS:
    print("\nExperiment comparison:")
    display(pd.DataFrame(compare_experiments(COMPARISON_RESULT_PATHS)))


In [ ]:
# ============================================================
# CELL 17 — Inspect raw observations / prompt debugging
# ============================================================

if result is None:
    print("No pipeline result. Run CELL 13 first.")
else:
    for i, item in enumerate(result["observations"], start=1):
        print("\n" + "=" * 100)
        print(
            f"{i}/{len(result['observations'])} | "
            f"{item.get('question_id')} | "
            f"{item.get('layer')} | "
            f"target={item.get('target')}"
        )
        print("-" * 100)
        print("QUESTION:")
        print(item.get("question"))
        print("\nRAW ANSWER:")
        print(item.get("raw_answer"))
        print("\nPARSED:")
        print(
            item.get("parsed_status")
            if item.get("layer") == "ATOMIC_FINDING"
            else item.get("parsed_answer")
        )
        print(
            "latency=", item.get("latency_seconds"),
            "| finish=", item.get("finish_reason"),
            "| truncated=", item.get("truncated"),
        )


In [ ]:
# ============================================================
# CELL 18 — Server diagnostics
# ============================================================
# Run this if model loading or inference fails.

log_path = Path(SERVER_LOG_PATH)
if log_path.is_file():
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(lines[-120:]))
else:
    print("No server log found at:", log_path)


In [ ]:
# ============================================================
# CELL 19 — Optional cleanup
# ============================================================
# Stop only when completely finished. Keep the server running during
# experiments so the model remains loaded in GPU memory.

# server.stop()
# print("DentalGPT server stopped.")
